# Mga recurrent neural network

Sa nakaraang module, tinalakay natin ang mayamang semantikong representasyon ng teksto. Ang arkitektura na ginagamit natin ay kumukuha ng pinagsama-samang kahulugan ng mga salita sa isang pangungusap, ngunit hindi nito isinasaalang-alang ang **ayusan** ng mga salita, dahil ang operasyon ng pagsasama na sumusunod sa embeddings ay inaalis ang impormasyong ito mula sa orihinal na teksto. Dahil ang mga modelong ito ay hindi kaya magrepresenta ng ayusan ng salita, hindi nila malulutas ang mas kumplikado o malabong mga gawain tulad ng paglikha ng teksto o pagsagot sa mga tanong.

Upang makuha ang kahulugan ng isang pagkakasunod-sunod ng teksto, gagamit tayo ng arkitektura ng neural network na tinatawag na **recurrent neural network**, o RNN. Kapag gumagamit ng RNN, ipinapasa natin ang pangungusap sa network isa-isa ang mga token, at gumagawa ang network ng ilang **estado**, na pagkatapos ay ipapasa natin muli sa network kasama ang susunod na token.

![Larawan na nagpapakita ng isang halimbawa ng pagbuo gamit ang recurrent neural network.](../../../../../translated_images/tl/rnn.27f5c29c53d727b5.webp)

Kung ibinigay ang input na pagkakasunod-sunod ng mga token na $X_0,\dots,X_n$, lumilikha ang RNN ng isang pagkakasunod-sunod ng mga bloke ng neural network, at sinasanay ang pagkakasunod-sunod na ito nang end-to-end gamit ang backpropagation. Ang bawat bloke ng network ay tumatanggap ng pares na $(X_i,S_i)$ bilang input, at gumagawa ng $S_{i+1}$ bilang resulta. Ang huling estado na $S_n$ o output na $Y_n$ ay pumapasok sa isang linear na classifier upang makabuo ng resulta. Ang lahat ng bloke ng network ay gumagamit ng parehong bigat, at sinasanay nang end-to-end gamit ang isang pass ng back propagation.

> Ipinapakita sa larawan sa itaas ang recurrent neural network sa unrolled na anyo (sa kaliwa), at sa mas compact na recurrent na representasyon (sa kanan). Mahalaga ring maunawaan na lahat ng RNN Cells ay may parehong **bigat na maaaring ibahagi**.

Dahil ang mga vector ng estado na $S_0,\dots,S_n$ ay ipinapasa sa network, kaya ng RNN na matutunan ang mga sunod-sunod na dependencies sa pagitan ng mga salita. Halimbawa, kapag lumitaw ang salitang *not* sa isang bahagi ng pagkakasunod-sunod, kaya nitong matutunang baligtarin ang ilang bahagi sa loob ng vector ng estado.

Sa loob, bawat RNN cell ay naglalaman ng dalawang matrix ng bigat: $W_H$ at $W_I$, at bias na $b$. Sa bawat hakbang ng RNN, kung ibinigay ang input na $X_i$ at estado ng input na $S_i$, kinakalkula ang output na estado bilang $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kung saan ang $f$ ay isang activation function (madalas ay $\tanh$).

> Para sa mga problema tulad ng paglikha ng teksto (na tatalakayin natin sa susunod na yunit) o pagsasaling makina gusto din natin makakuha ng ilang halaga ng output sa bawat hakbang ng RNN. Sa kasong ito, may isa pang matrix na $W_O$, at ang output ay kinakalkula bilang $Y_i=f(W_O\times S_i+b_O)$.

Tingnan natin kung paano makakatulong ang mga recurrent neural network sa pagklasipika ng ating dataset ng balita.

> Para sa sandbox na kapaligiran, kailangan nating patakbuhin ang sumusunod na cell upang matiyak na naka-install ang kinakailangang library, at na-prefetch ang data. Kung nagpapalakad ka nang lokal, maaari mong laktawan ang sumusunod na cell.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Kapag nagte-train ng malalaking modelo, maaaring maging problema ang alokasyon ng memorya ng GPU. Kailangan din nating mag-eksperimento sa iba’t ibang laki ng minibatch, upang magkasya ang data sa aming GPU memory, ngunit mabilis pa rin ang training. Kung pinapatakbo mo ang code na ito sa sarili mong GPU machine, maaari kang mag-eksperimento sa pag-aayos ng laki ng minibatch upang pabilisin ang training.

> **Note**: Kilala ang ilang bersyon ng NVidia drivers na hindi naglalabas ng memorya pagkatapos i-train ang modelo. Nagpapatakbo kami ng ilang halimbawa sa mga notebook na ito, at maaaring magdulot ito ng pagkauubos ng memorya sa ilang setup, lalo na kung gumagawa ka ng sariling mga eksperimento bilang bahagi ng parehong notebook. Kung makatagpo ka ng kakaibang mga error kapag nagsimulang mag-train ng modelo, maaari mong subukang i-restart ang kernel ng notebook.


In [3]:
batch_size = 16
embed_size = 64

## Simpleng RNN na tagapag-uri

Sa kaso ng simpleng RNN, bawat recurrent unit ay isang simpleng linear na network, na tumatanggap ng input vector at state vector, at gumagawa ng bagong state vector. Sa Keras, ito ay maaaring ipakita gamit ang `SimpleRNN` na layer.

Bagaman maaari nating direktang ipasa ang one-hot encoded tokens sa RNN layer, hindi ito magandang ideya dahil sa kanilang mataas na dimensionality. Kaya, gagamit tayo ng embedding layer upang mabawasan ang dimensionality ng mga word vectors, kasunod ng isang RNN layer, at panghuli ay isang `Dense` na tagapag-uri.

> **Tandaan**: Sa mga pagkakataon na hindi gaano kataas ang dimensionality, halimbawa kapag gumagamit ng character-level tokenization, maaaring makatuwiran na direktang ipasa ang one-hot encoded tokens sa RNN cell.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Tandaan:** Gumagamit kami ng hindi na-train na embedding layer dito para sa pagiging simple, ngunit para sa mas magagandang resulta maaari tayong gumamit ng pretrained embedding layer gamit ang Word2Vec, tulad ng ipinaliwanag sa nakaraang yunit. Magandang pagsasanay para sa iyo na i-adapt ang code na ito upang gumana sa pretrained embeddings.

Ngayon ay sanayin natin ang ating RNN. Ang mga RNN sa pangkalahatan ay medyo mahirap sanayin, dahil kapag na-unroll ang mga RNN cells sa haba ng sequence, ang bilang ng mga layer na sangkot sa backpropagation ay magiging napakalaki. Kaya kailangan nating pumili ng mas maliit na learning rate, at sanayin ang network sa mas malaking dataset upang makabuo ng magagandang resulta. Maaaring tumagal ito ng medyo matagal, kaya mas mainam na gumamit ng GPU.

Upang mapabilis ang proseso, tanging ang RNN model ay sasanayin lamang sa mga pamagat ng balita, hindi kabilang ang deskripsyon. Maaari mong subukang sanayin ito gamit ang deskripsyon at tingnan kung kaya mong mapagsanay ang modelo.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Tandaan** na malamang na mas mababa ang katumpakan dito, dahil nagte-training tayo gamit lamang ang mga pamagat ng balita.


## Muling pagtingin sa mga variable na sekwensya 

Tandaan na ang `TextVectorization` layer ay awtomatikong naglalagay ng padding sa mga sekwensya na may iba't ibang haba sa isang minibatch gamit ang pad tokens. Lumalabas na ang mga tokens na iyon ay nakikibahagi rin sa pagsasanay, at maaari itong maging sanhi ng komplikasyon sa pag-converge ng modelo.

May ilang mga paraan na maaari nating gawin upang mabawasan ang dami ng padding. Isa rito ay ang muling ayusin ang dataset ayon sa haba ng sekwensya at pagsamahin ang lahat ng sekwensya ayon sa laki. Ito ay maaaring gawin gamit ang `tf.data.experimental.bucket_by_sequence_length` function (tingnan ang [dokumentasyon](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Isa pang paraan ay ang paggamit ng **masking**. Sa Keras, ang ilang mga layer ay sumusuporta sa karagdagang input na nagpapakita kung alin sa mga token ang dapat isaalang-alang sa pagsasanay. Upang maisama ang masking sa ating modelo, maaari tayong magdagdag ng hiwalay na `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), o maaari nating tukuyin ang `mask_zero=True` parameter ng ating `Embedding` layer.

> **Tandaan**: Ang pagsasanay na ito ay tatagal ng humigit-kumulang 5 minuto upang matapos ang isang epoch sa buong dataset. Malaya kang itigil ang pagsasanay anumang oras kung mawalan ka ng pasensya. Maaari mo ring limitahan ang dami ng datos na gagamitin sa pagsasanay, sa pamamagitan ng pagdagdag ng `.take(...)` na pangungusap pagkatapos ng `ds_train` at `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Ngayon na gumagamit tayo ng masking, maaari nating sanayin ang modelo sa buong dataset ng mga pamagat at paglalarawan.

> **Tandaan**: Napansin mo ba na gumagamit tayo ng vectorizer na sinanay sa mga pamagat ng balita, at hindi sa buong katawan ng artikulo? Posibleng magdulot ito ng pagwawalang-bahala sa ilang mga token, kaya mas mabuting muling sanayin ang vectorizer. Gayunpaman, maaaring maliit lamang ang epekto nito, kaya mananatili tayo sa naunang pre-trained na vectorizer para sa kadalian.


## LSTM: Long short-term memory

Isa sa mga pangunahing problema ng RNNs ay ang **vanishing gradients**. Maaaring maging mahaba ang mga RNN, at maaaring mahirapan itong ipasa ang mga gradient pabalik hanggang sa unang layer ng network habang nagba-backpropagation. Kapag nangyari ito, hindi matututuhan ng network ang mga relasyon sa pagitan ng mga malalayong token. Isang paraan upang maiwasan ang problemang ito ay ang pagpapakilala ng **explicit state management** gamit ang **gates**. Ang dalawang pinakakaraniwang arkitektura na nagpapakilala ng mga gates ay ang **long short-term memory** (LSTM) at **gated relay unit** (GRU). Dito ay tatalakayin natin ang LSTMs.

![Image showing an example long short term memory cell](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Ang LSTM network ay inayos sa paraang kahalintulad ng RNN, ngunit may dalawang estado na ipinapasa mula layer sa layer: ang aktwal na estado na $c$, at ang nakatagong vector na $h$. Sa bawat yunit, ang nakatagong vector na $h_{t-1}$ ay pinagsasama sa input na $x_t$, at magkasama nilang kinokontrol kung ano ang mangyayari sa estado na $c_t$ at output na $h_{t}$ sa pamamagitan ng **gates**. Bawat gate ay may sigmoid activation (output na nasa saklaw na $[0,1]$), na maaaring isipin bilang isang bitwise mask kapag minultiplika sa state vector. Ang LSTMs ay may mga sumusunod na gates (mula kaliwa hanggang kanan sa larawan sa itaas):
* **forget gate** na tumutukoy kung aling mga bahagi ng vector na $c_{t-1}$ ang kailangan nating kalimutan, at kung alin ang ipapasa.
* **input gate** na tumutukoy kung gaano karaming impormasyon mula sa input vector at naunang nakatagong vector ang dapat isama sa state vector.
* **output gate** na kumukuha sa bagong state vector at nagpapasya kung alin sa mga bahagi nito ang gagamitin upang makabuo ng bagong nakatagong vector na $h_t$.

Ang mga bahagi ng estado na $c$ ay maaaring isipin bilang mga flag na maaaring i-on at i-off. Halimbawa, kapag nakatagpo tayo ng pangalang *Alice* sa sunod-sunod, huhulaan natin na ito ay tumutukoy sa isang babae, at itataas ang flag sa estado na nagsasabing mayroong pambabaeng pangngalan sa pangungusap. Kapag nakatagpo naman tayo ng salitang *and Tom*, itataas natin ang flag na nagsasabing mayroon tayong maramihang pangngalan. Sa ganitong paraan, sa pamamagitan ng pag-manipula ng estado, maaari nating subaybayan ang mga gramatikal na katangian ng pangungusap.

> **Note**: Narito ang isang mahusay na sanggunian para sa pag-unawa ng mga panloob na bahagi ng LSTMs: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) ni Christopher Olah.

Bagamat ang panloob na istruktura ng isang LSTM cell ay maaaring mukhang kumplikado, itinatago ng Keras ang implementasyong ito sa loob ng `LSTM` layer, kaya ang tanging kailangan nating gawin sa halimbawa sa itaas ay palitan ang recurrent layer:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Tandaan** na ang pagsasanay ng LSTMs ay medyo mabagal din, at maaaring hindi mo agad makita ang malaking pagtaas sa kawastuhan sa simula ng pagsasanay. Maaaring kailanganin mong ipagpatuloy ang pagsasanay ng ilang panahon upang makamit ang magandang kawastuhan.


## Bidirectional at multilayer na RNNs

Sa aming mga halimbawa hanggang ngayon, ang mga recurrent networks ay gumagana mula sa simula ng isang sequence hanggang sa dulo. Ito ay tila natural sa atin dahil sinusundan nito ang parehong direksyon kung saan tayo nagbabasa o nakikinig ng pananalita. Gayunpaman, para sa mga senaryo na nangangailangan ng random access ng input sequence, mas may saysay na patakbuhin ang recurrent computation sa dalawang direksyon. Ang mga RNN na nagpapahintulot ng computations sa dalawang direksyon ay tinatawag na **bidirectional** RNNs, at maaari silang malikha sa pamamagitan ng pagbalot ng recurrent layer gamit ang espesyal na `Bidirectonal` na layer.

> **Tandaan**: Ang `Bidirectional` layer ay gumagawa ng dalawang kopya ng layer sa loob nito, at itinatakda ang `go_backwards` property ng isa sa mga kopyang iyon sa `True`, na nagpapagalaw nito sa kabaligtarang direksyon kasabay ng sequence.

Ang mga recurrent networks, unidirectional man o bidirectional, ay nakakuha ng mga pattern sa loob ng isang sequence, at iniimbak ang mga ito sa mga state vector o ibinabalik bilang output. Tulad ng mga convolutional networks, maaari tayong bumuo ng isa pang recurrent layer pagkatapos ng una upang makuha ang mas mataas na antas ng mga pattern, na binuo mula sa mas mababang antas ng mga pattern na nakuha ng unang layer. Ito ay humahantong sa konsepto ng **multi-layer RNN**, na binubuo ng dalawa o higit pang mga recurrent network, kung saan ang output ng naunang layer ay ipinapasa sa susunod na layer bilang input.

![Larawan ng isang Multilayer long-short-term-memory- RNN](../../../../../translated_images/tl/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Larawan mula sa [napakagandang post na ito](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) ni Fernando López.*

Pinakapadali ng Keras ang paggawa ng mga network na ito, dahil kailangan mo lamang magdagdag ng mas maraming recurrent layer sa modelo. Para sa lahat ng layer maliban sa huli, kailangan nating tukuyin ang `return_sequences=True` parameter, dahil kailangan natin na ang layer ay magbalik ng lahat ng mga intermediate state, at hindi lang ang panghuling estado ng recurrent computation.

Gumawa tayo ng isang dalawang-layer na bidirectional LSTM para sa ating problema sa classificaton.

> **Tandaan** sa code na ito, muling tumatagal ng matagal bago matapos, ngunit nagbibigay ito ng pinakataas na accuracy na nakita natin hanggang ngayon. Kaya marahil ay sulit itong hintayin at tingnan ang resulta.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## Mga RNN para sa ibang mga gawain

Hanggang ngayon, nakatuon kami sa paggamit ng mga RNN para iklasipika ang mga sunud-sunod na teksto. Ngunit kaya nilang hawakan ang mas maraming gawain, tulad ng pagbuo ng teksto at pagsasalin ng makina &mdash; tatalakayin natin ang mga gawain na iyon sa susunod na bahagi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
